# Choosing the Optimal Data Management Architecture

You are the data engineer of a company that has to choose **one** of four architectures: Data Warehouse, Data Lake, Hybrid Data Lake or Lakehouse. In this exercise only **cost** counts: buying the data, storing it, moving it, querying it and maintaining the system. In real life cost is one factor among several, that is what the recommendation round is for.

**In class (group competition).** Each group calculates the 2-year cost of **one** architecture, the one it explained in the expert round. The numbers of all groups go on the whiteboard. Then every group recommends an architecture to the company, with one rule: **not your own.**

**Optional, at home.** Calculate all four architectures and check the whiteboard, plus a bonus task at the end. Nothing of this is required.

**How to work.** Task 1 is six small steps, one cell each. The comment next to every line says what to compute, you only write the formula. The first line of each step is already filled in as an example. Using an LLM is allowed.

In [ ]:
# Your group's architecture: "Data Warehouse", "Data Lake", "Hybrid Data Lake" or "Lakehouse"
MY_ARCHITECTURE = "Data Lake"

## Prices

| architecture | storage ($ per GB per month) | transfer ($ per GB) | queries ($ per 1,000 queries) | maintenance ($ per month) |
|---|---|---|---|---|
| Data Warehouse | 0.30 | 0.10 | 0.50 | 50 |
| Data Lake | 0.05 | 0.10 | 1.00 | 30 |
| Hybrid Data Lake | 0.35 | 0.10 | 0.50 | 70 |
| Lakehouse | 0.08 | 0.10 | 0.75 | 30 |

The same table as a DataFrame. `prices` is the row of your architecture, so `prices["storage_per_gb_month"]` is your storage price.

In [ ]:
import pandas as pd

costs = pd.DataFrame({
    "architecture":          ["Data Warehouse", "Data Lake", "Hybrid Data Lake", "Lakehouse"],
    "storage_per_gb_month":  [0.30, 0.05, 0.35, 0.08],
    "transfer_per_gb":       [0.10, 0.10, 0.10, 0.10],
    "query_per_1000":        [0.50, 1.00, 0.50, 0.75],
    "maintenance_per_month": [50, 30, 70, 30],
}).set_index("architecture")

prices = costs.loc[MY_ARCHITECTURE]
costs

## Usage over 2 years

**Time.** 2 years = 730 days = 104.3 weeks (730 / 7) = 24 months.

**Data sources.** Every GB has to be bought from the source (acquisition price below) and moved into your storage (transfer price from the table above).

| source | what | volume | acquisition price |
|---|---|---|---|
| A | web tracking | 10 GB per **day** | $0.20 per GB |
| B | shop backend export | 100 GB per **week** | $0.10 per GB |
| C | purchased financial market data feed | 20 GB per **month** | $20 per GB |

Source C bills per download: re-downloading old data costs the full price again. This matters in the extra question and in the bonus.

**Storage.** Storage starts empty and grows by 20 GB per month, nothing is deleted. It is billed every month on what is stored at that time: month 1 bills 20 GB, month 2 bills 40 GB, ..., month 24 bills 480 GB.

**Queries.**

| type | how often |
|---|---|
| 1 | 1,000 queries per **day** |
| 2 | 10,000 queries per **week** |
| 3 | 15,000 queries per **month** |

**Maintenance.** A fixed amount per month, from the price table.

In [ ]:
DAYS = 730          # 2 years
WEEKS = DAYS / 7    # 104.3
MONTHS = 24

## Task 1: the 2-year cost of your architecture (10 min)

Six steps, one cell each. Replace every `...` with a formula and run the cell.

In [ ]:
# Step 1: how many GB arrive from the sources in 2 years?
gb_a = 10 * DAYS       # source A: 10 GB per day
gb_b = ...             # source B: 100 GB per week
gb_c = ...             # source C: 20 GB per month
gb_total = gb_a + gb_b + gb_c

print(round(gb_total))

In [ ]:
# Step 2: acquisition, what you pay the sources for the data (the same for all architectures)
acquisition = ...      # gb_a times $0.20, plus gb_b times $0.10, plus gb_c times $20

print(round(acquisition))

In [ ]:
# Step 3: transfer, moving all that data into your storage
transfer = ...         # gb_total times prices["transfer_per_gb"]

print(round(transfer))

In [ ]:
# Step 4: storage, billed every month on the growing volume
gb_months = 20 * sum(range(1, MONTHS + 1))   # 20 + 40 + ... + 480 GB, one entry per month, summed up
storage = ...                                # gb_months times prices["storage_per_gb_month"]

print(round(gb_months))

In [ ]:
# Step 5: queries, the price is per 1,000 queries
n_queries = 1000 * DAYS + ...   # type 1 per day, plus type 2 per week, plus type 3 per month
queries = ...                   # n_queries / 1000 times prices["query_per_1000"]

print(round(n_queries))

In [ ]:
# Step 6: maintenance and the total
maintenance = ...      # MONTHS times prices["maintenance_per_month"]
total = acquisition + storage + transfer + queries + maintenance

print(f"{MY_ARCHITECTURE}: {total:.0f} $ in 2 years")
print(f"  acquisition {acquisition:.0f}, storage {storage:.0f}, transfer {transfer:.0f}, "
      f"queries {queries:.0f}, maintenance {maintenance:.0f}")

## Task 2: whiteboard

Write your total and the five components on the whiteboard. Acquisition and transfer must be the same for all groups. If not: who used a different assumption?

## Task 3: recommendation (2 min per group)

Based on the whiteboard table: which architecture would you recommend to the company? **Not your own.** Give one cost argument and one non-cost argument (performance, flexibility, risk of a data swamp, team skills, ...).

**Extra question.** After one year the schema of the cleaned tables changes and the whole history has to be rebuilt from raw data. Which architecture pays the financial data feed (source C) a **second time**, and which ones do not? Why?

## Optional: all four architectures

Steps 3 to 6 depend on the architecture, steps 1 and 2 do not. Put steps 3 to 6 into a function, loop over all four architectures, build one table (rows = architectures, columns = the five components and the total) and sort it by total. Compare with the whiteboard photo.

In [ ]:
def two_year_costs(architecture):
    p = costs.loc[architecture]
    # gb_total, gb_months, n_queries and acquisition from above do not depend on the architecture, reuse them
    storage = ...
    transfer = ...
    queries = ...
    maintenance = ...
    return {"acquisition": acquisition, "storage": storage, "transfer": transfer,
            "queries": queries, "maintenance": maintenance,
            "total": acquisition + storage + transfer + queries + maintenance}


comparison = pd.DataFrame({a: two_year_costs(a) for a in costs.index}).T.round(0)
comparison.sort_values("total")

## Bonus (optional): when things go wrong

Two things happen in the two years.

1. **Incident after 6 months.** An error in the transformation process destroys all cleaned data (staging and marts). There was no backup. You rebuild the cleaned data and, lesson learned, from month 7 on you keep a full copy of your storage: the storage bill doubles from month 7 to month 24.
2. **Schema update after 12 months.** The cleaned tables need a new schema and the whole history has to be rebuilt from raw data.

Rebuilding from raw data is free in our model if the raw data is still in your own storage (the transformation simply runs again). If it is **not** (a pure warehouse keeps only cleaned tables), the raw data has to be loaded from the sources again: transfer costs again and acquisition costs again (source C bills every download at full price, assume A and B do as well). The sources still have the history.

**Task.** Calculate the 2-year costs of all four architectures including both events. Which architecture became much more expensive, and why is that not visible in the price table at the top?

Hints: the extra storage is `20 * sum(range(7, MONTHS + 1))` GB-months. Re-loading `n` months of raw data costs `n / MONTHS` of the 2-year acquisition and transfer. Only the Data Warehouse has to re-load (6 months after the incident, 12 months after the schema update).

In [ ]:
# Your code here